<div style='background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);padding:50px 40px;border-radius:20px;margin-bottom:30px;box-shadow:0 20px 60px rgba(102,126,234,0.3);position:relative;overflow:hidden;'>
  <div style='position:absolute;top:-50px;right:-50px;width:200px;height:200px;background:rgba(255,255,255,0.05);border-radius:50%;'></div>
  <div style='position:relative;z-index:1;'>
    <div style='display:inline-block;background:rgba(255,255,255,0.15);padding:6px 16px;border-radius:20px;font-size:0.85em;color:rgba(255,255,255,0.9);margin-bottom:16px;'>📘 CookBook Series · Фаза A — Основа</div>
    <h1 style='color:white;font-size:2.8em;margin:0 0 12px 0;font-weight:700;letter-spacing:-0.02em;line-height:1.2;'>Урок 03: Mock-модель и Prefill</h1>
    <p style='color:rgba(255,255,255,0.85);font-size:1.15em;margin:0 0 24px 0;line-height:1.6;'>От синтетической модели к первому реальному forward pass — понимаем prefill-фазу изнутри.</p>
    <div style='display:flex;gap:20px;flex-wrap:wrap;'>
      <span style='color:rgba(255,255,255,0.7);font-size:0.9em;'>📅 2026-03-23</span>
      <span style='color:rgba(255,255,255,0.7);font-size:0.9em;'>⏱ 40 мин</span>
      <span style='color:rgba(255,255,255,0.7);font-size:0.9em;'>👤 @Verbasik</span>
      <span style='color:rgba(255,255,255,0.7);font-size:0.9em;'>🟡 Средний</span>
    </div>
  </div>
</div>

<div style='background:linear-gradient(to right,#f8f9ff,#ffffff);border:1px solid #e2e8f0;border-left:5px solid #667eea;padding:25px 30px;border-radius:0 16px 16px 0;margin-bottom:20px;box-shadow:0 4px 15px rgba(0,0,0,0.04);'>
  <h3 style='color:#667eea;margin:0 0 16px 0;font-size:1.3em;font-weight:700;'>📋 Содержание</h3>
  <ol style='color:#4a5568;line-height:2.2;margin:0;padding-left:20px;'>
    <li><a href='#section-intro' style='color:#667eea;text-decoration:none;'>Введение — что такое prefill</a></li>
    <li><a href='#section-setup' style='color:#667eea;text-decoration:none;'>Установка и импорты</a></li>
    <li><a href='#section-theory' style='color:#667eea;text-decoration:none;'>Теория: prefill vs decode</a></li>
    <li><a href='#section-config' style='color:#667eea;text-decoration:none;'>MockModelConfig</a></li>
    <li><a href='#section-mockmodel' style='color:#667eea;text-decoration:none;'>MockModel</a></li>
    <li><a href='#section-prefillout' style='color:#667eea;text-decoration:none;'>PrefillOutput</a></li>
    <li><a href='#section-runprefill' style='color:#667eea;text-decoration:none;'>run_prefill()</a></li>
    <li><a href='#section-tests' style='color:#667eea;text-decoration:none;'>Тесты и инварианты</a></li>
    <li><a href='#section-e2e' style='color:#667eea;text-decoration:none;'>End-to-end пайплайн (MockModel)</a></li>
    <li><a href='#section-prod' style='color:#667eea;text-decoration:none;'>Production Demo (Qwen3-8B)</a></li>
    <li><a href='#section-nanoinfer' style='color:#667eea;text-decoration:none;'>Соответствие nano-infer</a></li>
  </ol>
</div>

<div style='background:#ebf8ff;border-left:4px solid #63b3ed;padding:16px 20px;border-radius:0 10px 10px 0;margin:12px 0;color:#2d3748;'>
  <strong>⏱ Время чтения:</strong> ~40 мин &nbsp;·&nbsp; <strong>🎯 Цель:</strong> реализовать prefill-фазу с MockModel и Qwen3-8B &nbsp;·&nbsp; <strong>📋 Требования:</strong> Урок 01 (TokenizerAdapter), Урок 02 (SamplingParams)
</div>

<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-intro' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>📖</span>Введение — что такое Prefill</h2>

После того как мы научились **токенизировать текст** (Урок 01) и **семплировать токены** (Урок 02), пришло время соединить эти знания в единый поток выполнения.

**Prefill** — это первый и самый «дорогой» шаг генерации:

```
Prompt  →  Tokenize  →  Prefill  →  KV Cache
                                        ↓
                         Decode Loop (×N токенов)
                                        ↓
                              Detokenize / Stream
```

В этом уроке мы:
1. Реализуем **MockModel** — синтетическую LM для изучения интерфейсов без GPU
2. Напишем функцию **`run_prefill()`** — точку входа для первого forward pass
3. Запустим полный **E2E пайплайн** (токены → prefill → первый токен)
4. Покажем **production demo** с настоящей Qwen3-8B на GPU

<div style='background:#ebf8ff;border-left:4px solid #63b3ed;padding:16px 20px;border-radius:0 10px 10px 0;margin:12px 0;color:#2d3748;'><div style='font-weight:700;color:#2b6cb0;margin-bottom:6px;font-size:0.95em;'>📌 Определение</div><div style='line-height:1.7;'><strong>Prefill</strong> — первый forward pass по всем токенам промпта. Модель обрабатывает сразу весь входной текст и возвращает логиты для <em>следующего</em> токена (первого генерируемого). Побочный продукт — KV-кэш, который переиспользуется в decode loop.</div></div>


## Prefill в языковых моделях

### 1. Что такое Prefill и зачем он нужен

**Prefill** — это начальная фаза инференса языковой модели, на которой модель **одним проходом обрабатывает весь уже известный входной контекст** и строит внутренние представления, необходимые для дальнейшей генерации.

Иначе говоря, если пользователь подал в модель prompt длины $n$ токенов, то на этапе prefill модель не «придумывает» новые токены, а **читает** эти $n$ токенов, пропускает их через все слои трансформера и вычисляет:

* скрытые представления для каждой позиции;
* attention-связи между токенами контекста;
* ключи и значения attention, которые обычно сохраняются в **KV-cache**;
* логиты для последней позиции, откуда уже можно выбрать **первый генерируемый токен**.

Это можно кратко сформулировать так:

> **Prefill = “прочитать и закэшировать весь известный префикс”**
> **Decode = “порождать новые токены, используя уже построенный кэш”**

---

### 2. Интуиция

Представьте, что модель — это человек, который сначала читает весь абзац вопроса, запоминает контекст, выстраивает связи между словами, и только потом начинает отвечать.

Именно это и делает prefill:

* читает весь prompt;
* строит внутреннее понимание структуры текста;
* подготавливает состояние, из которого уже можно продолжить генерацию.

Без prefill модель не знает, **на каком контексте вообще нужно продолжать текст**.

---

### 3. Формальная постановка

Пусть входной prompt состоит из последовательности токенов:

$$
x_{1:n} = (x_1, x_2, \dots, x_n)
$$

Где:

* $x_i$ — токен на позиции $i$;
* $n$ — длина входного контекста в токенах;
* $x_{1:n}$ — весь prompt, который подаётся в модель.

Задача языковой модели — оценивать вероятности следующего токена:

$$
P(x_{n+1} \mid x_1, x_2, \dots, x_n)
$$

Где:

* $x_{n+1}$ — следующий токен, который модель должна предсказать;
* $P(x_{n+1} \mid x_1, \dots, x_n)$ — распределение вероятностей следующего токена при условии всего префикса.

Этап **prefill** — это вычисление всех промежуточных представлений, необходимых для получения этого распределения.

---

### 4. Что именно вычисляется на этапе Prefill

На входе трансформер получает эмбеддинги токенов с позиционной информацией:

$$
h_i^{(0)} = E(x_i) + p_i
$$

Где:

* $h_i^{(0)}$ — начальное скрытое представление токена на позиции $i$;
* $E(x_i)$ — embedding токена $x_i$;
* $p_i$ — позиционное представление для позиции $i$.

Далее на каждом слое self-attention строятся запросы, ключи и значения:

$$
q_i = W_Q h_i,\quad k_i = W_K h_i,\quad v_i = W_V h_i
$$

Где:

* $h_i$ — скрытое состояние токена на текущем слое;
* $q_i$ — query-вектор;
* $k_i$ — key-вектор;
* $v_i$ — value-вектор;
* $W_Q, W_K, W_V$ — обученные матрицы преобразования.

Attention для позиции $i$ вычисляется через все предыдущие позиции:

$$
\text{Attn}(q_i, K, V) = \sum_{j \le i} \alpha_{ij} v_j
$$

где веса внимания определяются как

$$
\alpha_{ij} = \frac{\exp\left(q_i^\top k_j / \sqrt{d_k}\right)}{\sum_{m \le i} \exp\left(q_i^\top k_m / \sqrt{d_k}\right)}
$$

Где:

* $\alpha_{ij}$ — вес внимания позиции $i$ к позиции $j$;
* $q_i^\top k_j$ — мера сходства запроса текущего токена с ключом предыдущего;
* $d_k$ — размерность key/query-векторов;
* ограничение $j \le i$ — каузальная маска: токен не может смотреть в будущее.

На этапе prefill это делается **для всех позиций prompt одновременно**.

---

### 5. Почему Prefill дорогой по вычислениям

Главная стоимость prefill возникает из-за self-attention по всему входному контексту.

Если длина prompt равна $n$, то каждая позиция может смотреть на все предыдущие. В грубой оценке это даёт квадратичную сложность по длине последовательности:

$$
\mathcal{O}(n^2)
$$

Где:

* $n$ — число токенов во входном prompt;
* $\mathcal{O}(n^2)$ — квадратичная стоимость attention по длине контекста.

Это означает:

* короткий prompt обрабатывается быстро;
* длинный prompt может быть очень дорогим;
* при больших контекстах prefill часто становится главным источником latency.

Именно поэтому длинные системные промпты, длинная история чата и большие вставленные документы так сильно влияют на скорость первого ответа.

---

### 6. KV-cache как главный результат Prefill

Самый важный практический результат prefill — это заполнение **KV-cache**.

Для каждого слоя и каждой позиции сохраняются ключи и значения:

$$
K^{(\ell)} = (k_1^{(\ell)}, k_2^{(\ell)}, \dots, k_n^{(\ell)}), \quad
V^{(\ell)} = (v_1^{(\ell)}, v_2^{(\ell)}, \dots, v_n^{(\ell)})
$$

Где:

* $\ell$ — индекс слоя трансформера;
* $k_i^{(\ell)}$ — key-вектор токена $i$ на слое $\ell$;
* $v_i^{(\ell)}$ — value-вектор токена $i$ на слое $\ell$;
* $K^{(\ell)}, V^{(\ell)}$ — наборы ключей и значений, накопленные для всего префикса.

Этот кэш сохраняется, чтобы на следующем этапе не пересчитывать весь prompt заново.

Именно поэтому prefill можно понимать как:

> **фазу построения и заполнения KV-cache по входному контексту**

---

### 7. Что происходит после Prefill: переход к Decode

Когда prefill завершён, модель уже знает всё о prompt и может начать генерацию.

Сначала из последнего скрытого состояния вычисляются логиты:

$$
z = W_{\text{out}} h_n
$$

Где:

* $h_n$ — скрытое состояние последней позиции prompt;
* $W_{\text{out}}$ — выходная матрица проекции в пространство словаря;
* $z$ — вектор логитов по всем токенам словаря.

После softmax получаем вероятности первого нового токена:

$$
P(w_i \mid x_{1:n}) = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}
$$

Где:

* $w_i$ — кандидатный следующий токен;
* $z_i$ — логит для токена $w_i$;
* $V$ — размер словаря.

После выбора первого нового токена начинается **decode**.

Теперь модель уже не обрабатывает весь prompt заново, а добавляет по одному новому токену, используя KV-cache из prefill.

---

### 8. Главное различие между Prefill и Decode

#### Prefill

* На вход подаётся **весь известный prompt**.
* Все токены контекста обрабатываются сразу.
* Attention считается по всему префиксу.
* Заполняется KV-cache.
* Получается распределение для **первого нового токена**.

#### Decode

* На вход подаётся уже **один новый токен за шаг**.
* Старый контекст не пересчитывается полностью.
* Используется KV-cache.
* На каждом шаге вычисляется следующий токен.
* Стоимость одного шага обычно близка к линейной по текущей длине контекста, а не квадратичной по всему prompt.

То есть:

$$
\text{Prefill} = \text{compute over prompt}
$$

$$
\text{Decode} = \text{incremental generation using cache}
$$

---

### 9. Почему время до первого токена зависит именно от Prefill

Пользователь часто замечает две разные характеристики скорости:

* **TTFT** — time to first token;
* **tokens/sec** — скорость дальнейшей генерации.

TTFT в большой степени определяется именно prefill, потому что до появления первого токена модель должна:

1. токенизировать prompt;
2. прогнать весь контекст через трансформер;
3. построить KV-cache;
4. вычислить логиты последней позиции;
5. выбрать первый токен.

Поэтому длинный prompt ухудшает именно **время до первого токена**, даже если дальнейшая генерация идёт быстро.

---

### 10. Числовая интуиция

Пусть у нас prompt длины:

$$
n = 4000
$$

токенов.

На этапе prefill модель должна обработать все эти 4000 позиций, а attention на каждом слое работает по матрице взаимодействий порядка

$$
4000 \times 4000 = 16,000,000
$$

пар позиций.

Где:

* $4000$ — длина контекста;
* $16,000,000$ — число пар взаимодействий в полном attention без учёта голов и батча.

Это не означает ровно столько операций во всей модели, но даёт интуицию, почему длинный контекст дорогой.

Если же decode генерирует следующий токен, то новый токен сравнивается уже с готовым кэшем, и нет необходимости пересчитывать всю матрицу attention для старых токенов.

---

### 11. Почему Prefill особенно важен в production-системах

В production LLM-сервисах prefill — одна из центральных точек оптимизации, потому что он влияет на:

* latency первого ответа;
* пропускную способность GPU;
* стоимость обработки длинных чатов и RAG-контекста;
* эффективность batching;
* объём памяти под KV-cache.

На практике инженерные вопросы вокруг prefill включают:

* как сокращать длину prompt;
* как эффективно паковать несколько запросов в batch;
* как шарить prompt-префиксы;
* как делать prefix caching;
* как ограничивать рост истории диалога;
* как балансировать между качеством ответа и стоимостью длинного контекста.

---

### 12. Prefill и prefix caching

Если у нескольких запросов есть общий префикс, то prefill для этой общей части можно не считать заново, а переиспользовать.

Например, если у всех запросов один и тот же system prompt, то можно закэшировать его KV-состояние и начинать prefill уже не с нуля, а с готового префикса.

Это называется **prefix caching**.

Идея такая:

$$
\text{KV}(\text{system prompt} + \text{shared context})
\rightarrow \text{reuse}
$$

Тогда при новом запросе нужно допрефиллить только уникальную часть prompt, а не весь контекст полностью.

Это резко снижает TTFT в сценариях, где большой кусок prompt повторяется между запросами.

---

### 13. Связь с обучением модели

Во время обучения модель тоже обрабатывает целые последовательности и учится предсказывать следующий токен в каждой позиции. Но в инференсе это разделяют на две практические фазы:

* **prefill** — обработка уже известного префикса;
* **decode** — пошаговое продолжение.

То есть prefill — это не отдельная “новая способность” модели, а инженерное имя для первой части автрорегрессионного инференса.

---

### 14. Типичные заблуждения

#### Заблуждение 1: Prefill — это генерация

Нет. Prefill сам по себе **не генерирует последовательность целиком**. Он подготавливает состояние, из которого можно начать генерацию.

#### Заблуждение 2: Prefill — это просто токенизация

Нет. Токенизация — это преобразование текста в токены. Prefill — это уже **прогон токенов через модель**.

#### Заблуждение 3: Prefill и decode стоят одинаково

Нет. Обычно prefill дорог по длинному контексту, а decode дорог по длине уже сгенерированного текста и числу шагов генерации.

#### Заблуждение 4: KV-cache нужен только для decode

Не совсем. KV-cache **создаётся именно во время prefill**, а затем используется в decode.

---

### 15. Короткая формула всей процедуры

Полный процесс инференса можно записать так:

$$
x_{1:n}
\xrightarrow{\text{prefill}}
\text{KV-cache},; P(x_{n+1}\mid x_{1:n})
\xrightarrow{\text{sample}}
x_{n+1}
\xrightarrow{\text{decode}}
x_{n+2}, x_{n+3}, \dots
$$

Где:

* $x_{1:n}$ — исходный prompt;
* $\text{KV-cache}$ — сохранённые ключи и значения attention для prompt;
* $P(x_{n+1}\mid x_{1:n})$ — распределение первого нового токена;
* $x_{n+1}$ — первый сэмплированный токен;
* decode — дальнейшая пошаговая генерация.

---

### Итог

**Prefill** — это начальная фаза инференса LLM, в которой модель **целиком обрабатывает уже известный входной контекст**, строит attention-представления, заполняет **KV-cache** и вычисляет распределение для **первого нового токена**. Главная особенность prefill в том, что он дорог по длине prompt и в большой степени определяет **время до первого токена**. Вся дальнейшая генерация опирается на результаты этой фазы.

<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-setup' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>📦</span>Установка и импорты</h2>


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐚 requirements · bash</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Bash</span></div>


In [ ]:
# Устанавливаем необходимые зависимости
# torch — основной тензорный фреймворк
# transformers — HuggingFace модели и токенизаторы
# matplotlib — визуализация логитов
!pip install torch transformers matplotlib numpy --quiet


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 imports.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
# ── Стандартная библиотека ──
import math
import warnings
from dataclasses import dataclass
from functools import lru_cache
from typing import List, Optional, Tuple

# ── Сторонние библиотеки: вычисления ──
import numpy as np
import torch

# ── Сторонние библиотеки: трансформеры ──
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Сторонние библиотеки: визуализация ──
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Константы проекта ──
# Vocab size Qwen3 — используем в MockModel и синтетических данных
QWEN3_VOCAB_SIZE: int = 151_936
# Идентификатор целевой production-модели
DEFAULT_MODEL_ID: str = "Qwen/Qwen3-8B"
# Кол-во top-токенов для визуализации
TOP_K_DISPLAY: int = 10
# Определяем доступное устройство (GPU или CPU)
DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Устройство: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")


<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-theory' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>📐</span>Теория: Prefill vs Decode</h2>

### Два режима авторегрессивной генерации

LLM-инференс состоит из **двух фаз**, которые фундаментально различаются:

| | Prefill | Decode |
|---|---|---|
| **Вход** | Все N токенов промпта | 1 новый токен |
| **Цель** | Первый логит + KV-кэш | Следующий логит |
| **Параллелизм** | Полный (все позиции сразу) | Последовательный |
| **Затраты** | O(N²) attention | O(N·kv_len) attention |
| **Частота** | 1 раз на запрос | M раз (кол-во новых токенов) |

### Что происходит внутри forward pass

Каждый слой трансформера выполняет:

1. **Self-Attention**: `Q·K^T / √d_k → Softmax → ·V` — токены «смотрят» друг на друга
2. **MLP/FFN**: два линейных слоя с активацией — нелинейное преобразование
3. **Layer Norm + Residual**: стабилизация обучения

После последнего слоя:

<div style='background:#faf5ff;border-left:4px solid #b794f4;padding:16px 20px;border-radius:0 10px 10px 0;margin:12px 0;color:#2d3748;'><div style='font-weight:700;color:#6b46c1;margin-bottom:6px;font-size:0.95em;'>📐 Ключевая формула</div><div style='line-height:1.7;'>$$\text{logits} = W_{\text{head}} \cdot h_{\text{last}}$$

Где:
- $W_{\text{head}}$ — матрица весов LM Head, размер $[V \times H]$;
- $h_{\text{last}}$ — скрытое состояние **последней позиции** последнего слоя, размер $[H]$;
- $\text{logits}$ — вектор размером $[V]$, где $V$ — размер словаря;
- Применяем **Softmax** к logits → вероятности → семплинг → первый новый токен.</div></div>

<div style='background:#fffbeb;border-left:4px solid #f59e0b;padding:16px 20px;border-radius:0 10px 10px 0;margin:12px 0;color:#2d3748;'><div style='font-weight:700;color:#b7791f;margin-bottom:6px;font-size:0.95em;'>💡 Совет</div><div style='line-height:1.7;'><strong>Зачем нужна MockModel?</strong> Реальная Qwen3-8B занимает ~16 GB VRAM. MockModel — синтетическая замена без attention и без GPU: позволяет изучить <em>форму данных и интерфейсы</em> на CPU за миллисекунды. Та же аналогия, что mock-объекты в unit-тестировании: мы тестируем наш код, а не веса модели.</div></div>


<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-config' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>⚙️</span>MockModelConfig</h2>

Конфигурация определяет все параметры синтетической модели. Используем `dataclass` — компактно и типизированно.


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 mock_model_config.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
@dataclass
class MockModelConfig:
    """
    Конфигурация синтетической языковой модели (MockModel).

    Description:
    ---------------
        Задаёт все параметры MockModel: размерности, тип данных и seed.
        Vocab size по умолчанию совпадает с Qwen3 — токены из реального
        токенизатора попадут в допустимый диапазон.

    Args:
    ---------------
        vocab_size: Размер словаря. По умолчанию = QWEN3_VOCAB_SIZE.
        hidden_size: Размерность скрытого представления (128 для CPU-демо).
        num_layers: Количество промежуточных линейных слоёв.
        dtype: Тип данных тензоров (float32 для CPU).
        seed: Фиксированное зерно — гарантирует воспроизводимость весов.
        device: Целевое устройство вычислений.

    Returns:
    ---------------
        Экземпляр конфигурации, готовый к передаче в MockModel.

    Raises:
    ---------------
        ValueError: Если hidden_size <= 0 или num_layers < 1.

    Examples:
    ---------------
        >>> cfg = MockModelConfig(vocab_size=100, hidden_size=32)
        >>> assert cfg.seed == 42
    """
    # Размер словаря — должен совпадать с токенизатором Qwen3
    vocab_size: int = QWEN3_VOCAB_SIZE
    # Скрытое измерение: маленькое значение ускоряет CPU-демо
    hidden_size: int = 128
    # Количество линейных слоёв между embedding и lm_head
    num_layers: int = 2
    # Тип данных: float32 стабилен на CPU
    dtype: torch.dtype = torch.float32
    # Зерно генератора: обеспечивает одинаковые веса при каждом запуске
    seed: int = 42
    # Устройство: 'cpu' для учебного режима, 'cuda' для GPU
    device: str = "cpu"

    def __post_init__(self) -> None:
        """Валидирует параметры после инициализации dataclass."""
        # Скрытое измерение должно быть положительным
        if self.hidden_size <= 0:
            raise ValueError(
                f"hidden_size должен быть > 0, получено {self.hidden_size}"
            )
        # Хотя бы один слой обязателен
        if self.num_layers < 1:
            raise ValueError(
                f"num_layers >= 1, получено {self.num_layers}"
            )


# Быстрая проверка конфигурации
_cfg_test = MockModelConfig(vocab_size=100, hidden_size=16, num_layers=1)
assert _cfg_test.seed == 42
print("✅ MockModelConfig: OK")
print(f"   vocab_size={_cfg_test.vocab_size}, hidden_size={_cfg_test.hidden_size}")

<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-mockmodel' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>🏗️</span>MockModel</h2>

<div style='background:#ffffff;border:1px solid #e2e8f0;border-radius:14px;padding:24px 28px;margin:20px 0;box-shadow:0 2px 8px rgba(0,0,0,0.04);'><div style='display:flex;align-items:center;gap:10px;margin-bottom:16px;'><div style='width:32px;height:32px;background:linear-gradient(135deg,#667eea,#764ba2);border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:0.9em;'>🔬</div><h4 style='margin:0;color:#2d3748;font-size:1.15em;font-weight:700;'>Архитектура MockModel</h4></div><div style='color:#4a5568;line-height:1.8;font-size:0.97em;'>MockModel — <em>минимальная</em> языковая модель: <strong>Embedding → [Linear + ReLU] × N → LM Head</strong>. Attention здесь намеренно отсутствует — нам нужна только правильная <em>форма данных</em> (input_ids → logits) и детерминированность. Это позволяет запускать тесты и E2E пайплайн на CPU за миллисекунды, не загружая 16 GB весов Qwen3-8B. В Lesson-08 мы подставим реальную модель — интерфейс останется тем же.</div></div>


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 mock_model.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
class MockModel:
    """
    Синтетическая языковая модель для изучения prefill-интерфейса.

    Description:
    ---------------
        Архитектура: Embedding(vocab, H) → [Linear(H,H) + ReLU] x N → Linear(H, vocab).
        Работает полностью на CPU, детерминирована при фиксированном seed.
        Не содержит attention — каждая позиция обрабатывается независимо.
        Форма forward: [B, S] → [B, S, V].
        Форма prefill: [B, S] → [B, V]  (только последняя позиция).

    Args:
    ---------------
        config: Конфигурация MockModelConfig.

    Returns:
    ---------------
        Инициализированная модель с детерминированными весами.

    Raises:
    ---------------
        TypeError: Если input_ids не является torch.Tensor.
        ValueError: Если токены >= vocab_size или последовательность пустая.

    Examples:
    ---------------
        >>> cfg = MockModelConfig(vocab_size=100, hidden_size=16, num_layers=1)
        >>> m = MockModel(cfg)
        >>> logits = m.forward(torch.tensor([[1, 2, 3]]))
        >>> assert logits.shape == (1, 3, 100)
    """

    def __init__(self, config: MockModelConfig) -> None:
        # Сохраняем конфигурацию для последующих проверок
        self.config: MockModelConfig = config

        # Инициализируем генератор с фиксированным seed — воспроизводимость весов
        gen: torch.Generator = torch.Generator()
        gen.manual_seed(config.seed)

        # Embedding матрица: [vocab_size, hidden_size]
        # Значения в [-0.1, 0.1] — небольшой диапазон для стабильных логитов
        self._emb: torch.Tensor = torch.empty(
            config.vocab_size, config.hidden_size, dtype=config.dtype
        ).uniform_(-0.1, 0.1, generator=gen)

        # Промежуточные слои: каждый [hidden_size, hidden_size]
        # Инициализация по Хе — подходит для ReLU-активации
        self._w: List[torch.Tensor] = []
        self._b: List[torch.Tensor] = []
        scale: float = math.sqrt(2.0 / config.hidden_size)
        for _ in range(config.num_layers):
            self._w.append(
                torch.empty(config.hidden_size, config.hidden_size,
                            dtype=config.dtype).normal_(0.0, scale, generator=gen)
            )
            # Смещения инициализируем нулями
            self._b.append(torch.zeros(config.hidden_size, dtype=config.dtype))

        # LM Head: [vocab_size, hidden_size] — проекция скрытого → словарь
        lm_scale: float = math.sqrt(2.0 / config.hidden_size)
        self._lm_head: torch.Tensor = torch.empty(
            config.vocab_size, config.hidden_size, dtype=config.dtype
        ).normal_(0.0, lm_scale, generator=gen)

    # ── forward ─────────────────────────────────────────────────────
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        '''
        Выполняет полный forward pass по последовательности токенов.

        Description:
        ---------------
            Применяет Embedding → Linear × N → LM Head к каждой позиции.
            Нет attention: позиции независимы друг от друга.

        Args:
        ---------------
            input_ids: Целочисленный тензор [B, S] или [S].

        Returns:
        ---------------
            Логиты [B, S, V] или [S, V] для каждой позиции.

        Raises:
        ---------------
            TypeError: Если input_ids не torch.Tensor.
            ValueError: Если max(input_ids) >= vocab_size.

        Examples:
        ---------------
            >>> ids = torch.tensor([[0, 1, 2]])    # [1, 3]
            >>> logits = model.forward(ids)
            >>> assert logits.shape == (1, 3, model.config.vocab_size)
        '''
        # Проверяем тип входного тензора
        if not isinstance(input_ids, torch.Tensor):
            raise TypeError(
                f"input_ids должен быть torch.Tensor, получено {type(input_ids)}"
            )
        # Проверяем диапазон токенов
        if int(input_ids.max()) >= self.config.vocab_size:
            raise ValueError(
                f"Токен {int(input_ids.max())} >= vocab_size={self.config.vocab_size}"
            )

        # Запоминаем исходную размерность для восстановления
        was_1d: bool = (input_ids.dim() == 1)
        if was_1d:
            input_ids = input_ids.unsqueeze(0)   # [S] → [1, S]

        # === Шаг 1: Embedding lookup — каждый токен → вектор ===
        # [B, S] → [B, S, H]  (индексирование строк embedding матрицы)
        hidden: torch.Tensor = self._emb[input_ids]

        # === Шаг 2: Промежуточные линейные слои с ReLU ===
        for weight, bias in zip(self._w, self._b):
            # Линейное преобразование: [B, S, H] × [H, H]^T + [H] → [B, S, H]
            hidden = hidden @ weight.T + bias
            # ReLU — нелинейная активация (обнуляет отрицательные значения)
            hidden = torch.relu(hidden)

        # === Шаг 3: LM Head — проекция скрытого в пространство словаря ===
        # [B, S, H] × [V, H]^T → [B, S, V]
        logits: torch.Tensor = hidden @ self._lm_head.T

        # Восстанавливаем исходную размерность если вход был 1D
        if was_1d:
            logits = logits.squeeze(0)   # [1, S, V] → [S, V]

        return logits

    # ── prefill ─────────────────────────────────────────────────────
    def prefill(self, input_ids: torch.Tensor) -> torch.Tensor:
        '''
        Выполняет prefill — возвращает логиты только последней позиции.

        Description:
        ---------------
            Вызывает forward() и извлекает логиты позиции [-1] по оси seq_len.
            Именно этот вектор используется для семплинга первого нового токена.

        Args:
        ---------------
            input_ids: Тензор [B, S] или [S]. S >= 1.

        Returns:
        ---------------
            Логиты [B, V] или [V] — предсказание следующего токена.

        Raises:
        ---------------
            ValueError: Если input_ids пустой (numel == 0).

        Examples:
        ---------------
            >>> ids = torch.tensor([[1, 2, 3, 4, 5]])   # [1, 5]
            >>> last = model.prefill(ids)
            >>> assert last.shape == (1, model.config.vocab_size)
        '''
        # Проверяем непустую последовательность
        if input_ids.numel() == 0:
            raise ValueError("input_ids не может быть пустым")

        # Определяем и нормализуем форму входа
        was_1d: bool = (input_ids.dim() == 1)
        if was_1d:
            input_ids = input_ids.unsqueeze(0)   # [S] → [1, S]

        # Выполняем полный forward pass: [B, S] → [B, S, V]
        logits_all: torch.Tensor = self.forward(input_ids)

        # Берём ПОСЛЕДНЮЮ позицию по оси dim=1 (seq_len), не по dim=0 (batch)!
        # [B, S, V] → [B, V]
        last_logits: torch.Tensor = logits_all[:, -1, :]

        # Восстанавливаем 1D формат если вход был 1D
        if was_1d:
            last_logits = last_logits.squeeze(0)   # [1, V] → [V]

        return last_logits


# Быстрый smoke-test
_m = MockModel(MockModelConfig(vocab_size=50, hidden_size=8, num_layers=1))
_ids = torch.tensor([[1, 2, 3]])
assert _m.forward(_ids).shape == (1, 3, 50)
assert _m.prefill(_ids).shape == (1, 50)
print("✅ MockModel: OK")
print(f"   forward([1,3]) → {_m.forward(_ids).shape}")
print(f"   prefill([1,3]) → {_m.prefill(_ids).shape}")


<div style='background:#fff5f5;border-left:4px solid #fc8181;padding:16px 20px;border-radius:0 10px 10px 0;margin:12px 0;color:#2d3748;'><div style='font-weight:700;color:#c53030;margin-bottom:6px;font-size:0.95em;'>⚠️ Внимание</div><div style='line-height:1.7;'><code>logits_all[:, -1, :]</code> — индекс <code>-1</code> по оси <strong>dim=1</strong> (seq_len), не по dim=0 (batch)! Ошибка <code>logits_all[-1]</code> взяла бы <em>последний пример батча</em>, а не последний токен — классический источник багов при первом знакомстве с prefill.</div></div>


<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-prefillout' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>📦</span>PrefillOutput</h2>

Структура данных для результата prefill-фазы. В реальном движке здесь также хранился бы handle на KV-кэш — добавим в Lesson-05.


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 prefill_output.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
@dataclass
class PrefillOutput:
    '''
    Результат prefill-фазы.

    Description:
    ---------------
        Содержит логиты последней позиции и длину входной последовательности.
        В реальном движке (nano-infer) здесь также хранится kv_cache_handle —
        указатель на сохранённые K, V для последующего decode loop.
        KV cache будет добавлен в Lesson-05.

    Args:
    ---------------
        logits: Тензор [B, V] — логиты следующего токена для каждого примера.
        input_len: Длина входной последовательности (количество токенов промпта).

    Returns:
    ---------------
        Структура с результатами prefill, готовая к передаче в sampler.

    Raises:
    ---------------
        нет.

    Examples:
    ---------------
        >>> out = PrefillOutput(logits=torch.zeros(1, 100), input_len=5)
        >>> assert out.input_len == 5
    '''
    # Логиты для предсказания следующего токена: [B, V]
    logits: torch.Tensor
    # Длина входной последовательности S (количество токенов промпта)
    input_len: int
    # NOTE: kv_cache_handle — добавим в Lesson-05 (KV Cache)

print("✅ PrefillOutput: OK")


<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-runprefill' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>🚀</span>run_prefill()</h2>

Главная функция раздела — принимает модель и токены, возвращает `PrefillOutput`. Это аналог `Scheduler._forward()` из nano-infer при `phase='prefill'`.


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 run_prefill.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
def run_prefill(
    model: MockModel,
    input_ids: torch.Tensor,
) -> PrefillOutput:
    """
    Выполняет prefill-фазу: forward pass по промпту → логиты первого нового токена.

    Description:
    ---------------
        Точка входа для planировщика перед началом decode loop.
        Нормализует вход до [B, S], вызывает model.prefill(), оборачивает в PrefillOutput.

    Args:
    ---------------
        model: Синтетическая модель MockModel.
        input_ids: Токены промпта формы [B, S] или [S]. B >= 1, S >= 1.

    Returns:
    ---------------
        PrefillOutput(logits=[B, V], input_len=S).

    Raises:
    ---------------
        TypeError: Если model не MockModel.
        ValueError: Если input_ids пустой.

    Examples:
    ---------------
        >>> cfg = MockModelConfig(vocab_size=100, hidden_size=16, num_layers=1)
        >>> m = MockModel(cfg)
        >>> out = run_prefill(m, torch.tensor([[1, 2, 3]]))
        >>> assert out.logits.shape == (1, 100) and out.input_len == 3
    """
    # Проверяем тип модели — явная проверка помогает поймать ошибки рано
    if not isinstance(model, MockModel):
        raise TypeError(f"model должен быть MockModel, получено {type(model)}")
    # Проверяем непустой вход
    if input_ids.numel() == 0:
        raise ValueError("input_ids не может быть пустым")

    # Нормализуем вход: гарантируем batch dimension [B, S]
    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)   # [S] → [1, S]

    # Запоминаем длину входа ДО forward pass
    input_len: int = input_ids.shape[1]   # S

    # Выполняем prefill: [B, S] → [B, V]  (только последняя позиция)
    logits: torch.Tensor = model.prefill(input_ids)

    # Возвращаем структурированный результат
    return PrefillOutput(logits=logits, input_len=input_len)


# Smoke-test
_cfg = MockModelConfig(vocab_size=200, hidden_size=16, num_layers=1)
_m2 = MockModel(_cfg)
_out = run_prefill(_m2, torch.tensor([[10, 20, 30]]))
assert _out.logits.shape == (1, 200)
assert _out.input_len == 3
print("✅ run_prefill: OK")
print(f"   logits.shape={_out.logits.shape}, input_len={_out.input_len}")


<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-tests' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>🔬</span>Тесты и инварианты</h2>

Четыре теста проверяют ключевые свойства MockModel и run_prefill. Все они должны выполняться за < 1 секунды на CPU.


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 test_shape.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
def test_shape() -> None:
    '''
    Проверяет форму выходных логитов для разных комбинаций [B, S].

    Description:
    ---------------
        prefill([B, S]) должен всегда возвращать [B, V], независимо от S.
        input_len должен совпадать с S.

    Args:
    ---------------
        нет — тест создаёт данные внутри.

    Returns:
    ---------------
        None. Выбрасывает AssertionError при нарушении инварианта формы.

    Raises:
    ---------------
        AssertionError: Если форма логитов не совпадает с [B, V].

    Examples:
    ---------------
        >>> test_shape()
        ✅ test_shape: PASSED
    '''
    # Маленькая конфигурация для быстрого теста
    cfg = MockModelConfig(vocab_size=100, hidden_size=16, num_layers=1, seed=0)
    m = MockModel(cfg)

    # Тест 1: батч [B=2, S=5] → ожидаем logits [2, 100], input_len=5
    ids_2x5 = torch.randint(0, 100, (2, 5))
    out = run_prefill(m, ids_2x5)
    assert out.logits.shape == (2, 100), f"Ожидали (2,100), получили {out.logits.shape}"
    assert out.input_len == 5

    # Тест 2: одиночный пример [B=1, S=3] → logits [1, 100], input_len=3
    ids_1x3 = torch.randint(0, 100, (1, 3))
    out2 = run_prefill(m, ids_1x3)
    assert out2.logits.shape == (1, 100)
    assert out2.input_len == 3

    # Тест 3: длинная последовательность [B=1, S=64]
    ids_long = torch.randint(0, 100, (1, 64))
    out3 = run_prefill(m, ids_long)
    assert out3.logits.shape == (1, 100)
    assert out3.input_len == 64

    print("✅ test_shape: PASSED")
    for o, label in [(out, "[2,5]"), (out2, "[1,3]"), (out3, "[1,64]")]:
        print(f"   {label} → logits{list(o.logits.shape)}, input_len={o.input_len}")

test_shape()


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 test_determinism.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
def test_determinism() -> None:
    '''
    Проверяет, что одинаковый вход всегда даёт одинаковые логиты.

    Description:
    ---------------
        MockModel детерминирована: нет dropout, нет random в forward.
        Два вызова с одинаковым input_ids должны возвращать побитово равные логиты.

    Args:
    ---------------
        нет.

    Returns:
    ---------------
        None.

    Raises:
    ---------------
        AssertionError: Если два вызова дают разные логиты.

    Examples:
    ---------------
        >>> test_determinism()
        ✅ test_determinism: PASSED
    '''
    cfg = MockModelConfig(vocab_size=200, hidden_size=32, num_layers=2, seed=7)
    m = MockModel(cfg)
    ids = torch.tensor([[10, 20, 30, 40, 50]])   # фиксированный ввод

    # Два независимых вызова должны дать ИДЕНТИЧНЫЕ результаты
    out1 = run_prefill(m, ids)
    out2 = run_prefill(m, ids)

    # torch.equal проверяет побитовое равенство
    assert torch.equal(out1.logits, out2.logits),             "Логиты отличаются — нарушена детерминированность!"

    print("✅ test_determinism: PASSED")
    diff = (out1.logits - out2.logits).abs().max().item()
    print(f"   Max абсолютная разница между вызовами: {diff:.2e}")

test_determinism()


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 test_batch_independence.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
def test_batch_independence() -> None:
    '''
    Проверяет, что результат seq_A не зависит от других примеров в батче.

    Description:
    ---------------
        MockModel не использует cross-attention между примерами батча.
        prefill([[a,b,c]]) должен совпадать с prefill([[a,b,c],[x,y,z]])[0].
        Этот инвариант верен и для реальных transformer-моделей.

    Args:
    ---------------
        нет.

    Returns:
    ---------------
        None.

    Raises:
    ---------------
        AssertionError: Если добавление второго примера меняет логиты первого.

    Examples:
    ---------------
        >>> test_batch_independence()
        ✅ test_batch_independence: PASSED
    '''
    cfg = MockModelConfig(vocab_size=50, hidden_size=16, num_layers=1, seed=1)
    m = MockModel(cfg)

    # Две последовательности одинаковой длины (для удобства cat)
    seq_a = torch.tensor([[1, 2, 3, 4]])   # [1, 4]
    seq_b = torch.tensor([[5, 6, 7, 8]])   # [1, 4]

    # seq_a в одиночку
    out_alone = run_prefill(m, seq_a)

    # seq_a и seq_b вместе в батче [2, 4]
    batch = torch.cat([seq_a, seq_b], dim=0)
    out_batch = run_prefill(m, batch)

    # Логиты seq_a должны совпадать — в обоих случаях
    assert torch.allclose(out_alone.logits[0], out_batch.logits[0], atol=1e-5),             "Логиты seq_A изменились при добавлении seq_B в батч!"

    print("✅ test_batch_independence: PASSED")
    diff = (out_alone.logits[0] - out_batch.logits[0]).abs().max().item()
    print(f"   seq_A alone vs seq_A in batch — max diff: {diff:.2e}")

test_batch_independence()


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 test_single_vs_batch.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
def test_single_vs_batch_consistency() -> None:
    '''
    Проверяет согласованность 1D и 2D форматов входа.

    Description:
    ---------------
        model.prefill(ids_1d) и run_prefill(model, ids_1d.unsqueeze(0))
        должны давать одинаковые результаты.
        Это гарантирует корректность нормализации размерности в run_prefill.

    Args:
    ---------------
        нет.

    Returns:
    ---------------
        None.

    Raises:
    ---------------
        AssertionError: Если 1D и 2D форматы дают разные результаты.

    Examples:
    ---------------
        >>> test_single_vs_batch_consistency()
        ✅ test_single_vs_batch_consistency: PASSED
    '''
    cfg = MockModelConfig(vocab_size=80, hidden_size=24, num_layers=1, seed=99)
    m = MockModel(cfg)

    # 1D вход: [S=6]
    ids_1d = torch.tensor([10, 20, 30, 40, 50, 60])
    # 2D вход: [B=1, S=6]
    ids_2d = ids_1d.unsqueeze(0)

    # model.prefill напрямую
    logits_direct_1d = m.prefill(ids_1d)        # [V]
    logits_direct_2d = m.prefill(ids_2d)        # [1, V]

    # run_prefill через wrapper
    out_via_run = run_prefill(m, ids_2d)        # PrefillOutput, logits [1, V]

    # Все результаты должны совпадать
    assert torch.allclose(logits_direct_1d, logits_direct_2d.squeeze(0), atol=1e-6)
    assert torch.allclose(logits_direct_2d, out_via_run.logits, atol=1e-6)

    print("✅ test_single_vs_batch_consistency: PASSED")
    print(f"   prefill(1D) → {logits_direct_1d.shape}")
    print(f"   prefill(2D) → {logits_direct_2d.shape}")
    print(f"   run_prefill → {out_via_run.logits.shape}")

test_single_vs_batch_consistency()


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 run_all_tests.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
def run_all_tests() -> None:
    '''
    Запускает все 4 теста инвариантов MockModel и run_prefill.

    Description:
    ---------------
        Последовательно вызывает все тест-функции.
        Если любой тест не прошёл — выбрасывается AssertionError.

    Args:
    ---------------
        нет.

    Returns:
    ---------------
        None.

    Raises:
    ---------------
        AssertionError: При нарушении любого инварианта.

    Examples:
    ---------------
        >>> run_all_tests()
        ✅ Все тесты пройдены!
    '''
    print("🧪 Запуск всех тестов MockModel + run_prefill...\n")
    test_shape()
    print()
    test_determinism()
    print()
    test_batch_independence()
    print()
    test_single_vs_batch_consistency()
    print("\n" + "=" * 55)
    print("✅ Все тесты пройдены! MockModel и run_prefill работают корректно.")

run_all_tests()


<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-e2e' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>🔄</span>End-to-end пайплайн (MockModel)</h2>

Соберём всё вместе: синтетические токены → prefill → семплинг первого токена.
Без GPU, без загрузки реальной модели — только интерфейсы.


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 e2e_setup.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
# Синтетические токены промпта «Что такое KV-кэш?» (примерные Qwen3 token IDs)
# В реальности токены генерирует TokenizerAdapter из Урока 01
DEMO_TOKENS: List[int] = [
    151644,  # <|im_start|>
    872,     # «user»
    198,     # newline
    102853,  # «Что»
    99355,   # «такое»
    1967,    # «KV»
    12,      # «-»
    31708,   # «кэш»
    30,      # «?»
    151645,  # <|im_end|>
]
print(f"Длина промпта: {len(DEMO_TOKENS)} токенов")
print(f"Token IDs: {DEMO_TOKENS}")


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 e2e_pipeline.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
def run_e2e_mock_demo(
    prompt_token_ids: List[int],
    temperature: float = 1.0,
    top_k: int = 50,
    seed: int = 42,
) -> Tuple[int, PrefillOutput]:
    '''
    Полный E2E пайплайн с MockModel: токены → prefill → первый токен.

    Description:
    ---------------
        Объединяет MockModel (этот урок) и примитивы семплинга (Урок 02):
        1. Создаёт MockModel с vocab_size = QWEN3_VOCAB_SIZE.
        2. Выполняет run_prefill() → логиты [1, V].
        3. Применяет temperature + top-k фильтрацию.
        4. Семплирует первый токен.

    Args:
    ---------------
        prompt_token_ids: Список целых — токены промпта.
        temperature: Температура (0 = greedy, > 0 = стохастично).
        top_k: Количество токенов для top-k фильтрации (0 = без фильтра).
        seed: Зерно для воспроизводимости семплинга.

    Returns:
    ---------------
        Кортеж (first_token_id: int, PrefillOutput).

    Raises:
    ---------------
        ValueError: Если prompt_token_ids пустой.

    Examples:
    ---------------
        >>> token, out = run_e2e_mock_demo([1, 2, 3])
        >>> assert isinstance(token, int)
    '''
    # Проверяем непустой промпт
    if not prompt_token_ids:
        raise ValueError("prompt_token_ids не может быть пустым")

    # Шаг 1: инициализируем MockModel с vocab=QWEN3_VOCAB_SIZE
    cfg = MockModelConfig(
        vocab_size=QWEN3_VOCAB_SIZE,
        hidden_size=128,
        num_layers=2,
        seed=seed,
    )
    model = MockModel(cfg)
    print(f"✅ MockModel: vocab={cfg.vocab_size}, hidden={cfg.hidden_size}")

    # Шаг 2: формируем input_ids тензор [B=1, S]
    input_ids: torch.Tensor = torch.tensor([prompt_token_ids], dtype=torch.long)
    print(f"📝 input_ids: {input_ids.shape}  "
          f"(B={input_ids.shape[0]}, S={input_ids.shape[1]})")

    # Шаг 3: prefill — первый forward pass → логиты [1, V]
    prefill_out: PrefillOutput = run_prefill(model, input_ids)
    print(f"⚙️  Prefill → logits{list(prefill_out.logits.shape)}")

    # Шаг 4: семплинг первого нового токена
    # Берём логиты первого (единственного) примера батча
    logits: torch.Tensor = prefill_out.logits[0]   # [V]

    if temperature <= 0.0:
        # Greedy: просто argmax
        first_token: int = int(torch.argmax(logits).item())
        print(f"🎯 Greedy → token_id={first_token}")
        return first_token, prefill_out

    # Temperature scaling: логиты / T  (большой T → более равномерное распределение)
    scaled: torch.Tensor = logits / max(temperature, 1e-8)

    # Top-k фильтрация: оставляем только k наибольших логитов
    if top_k > 0:
        k_actual: int = min(top_k, scaled.numel())
        # Порог: k-й по величине логит
        kth_val, _ = torch.topk(scaled, k_actual)
        min_thresh: float = kth_val[-1].item()
        # Всё ниже порога → -inf (нулевая вероятность)
        scaled = scaled.masked_fill(scaled < min_thresh, float("-inf"))

    # Softmax → вероятности → мультиномиальное семплирование
    gen = torch.Generator()
    gen.manual_seed(seed)
    probs: torch.Tensor = torch.softmax(scaled, dim=-1)   # [V]
    first_token = int(torch.multinomial(probs, num_samples=1, generator=gen).item())

    print(f"🎲 Семплинг (T={temperature}, top_k={top_k}) → token_id={first_token}")
    print(f"   P(token)={probs[first_token].item():.4%}")

    return first_token, prefill_out


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 run_e2e_demo.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
print("=" * 60)
print("🚀 E2E Пайплайн: MockModel + Prefill + Семплинг")
print("=" * 60)

# Режим 1: greedy (temperature=0)
print("\n--- Greedy (T=0) ---")
t_greedy, out_greedy = run_e2e_mock_demo(DEMO_TOKENS, temperature=0.0)

# Режим 2: temperature sampling
print("\n--- Temperature (T=1.0, top_k=50) ---")
t_sample, out_sample = run_e2e_mock_demo(DEMO_TOKENS, temperature=1.0, top_k=50)

# Режим 3: высокая температура — более случайные выборы
print("\n--- Высокая T (T=2.0, top_k=200) ---")
t_hot, out_hot = run_e2e_mock_demo(DEMO_TOKENS, temperature=2.0, top_k=200)

print("\n" + "=" * 60)
print(f"Greedy:   token_id = {t_greedy}")
print(f"Sampled:  token_id = {t_sample}")
print(f"Hot:      token_id = {t_hot}")
print(f"input_len = {out_greedy.input_len} токенов")
print("✅ E2E пайплайн завершён успешно!")


<div style='background:#f0fff4;border-left:4px solid #48bb78;padding:16px 20px;border-radius:0 10px 10px 0;margin:12px 0;color:#2d3748;'><div style='font-weight:700;color:#276749;margin-bottom:6px;font-size:0.95em;'>✅ Результат</div><div style='line-height:1.7;'>E2E пайплайн работает без GPU и без загрузки реальной модели. MockModel возвращает те же <em>формы тензоров</em>, что и Qwen3-8B: <code>PrefillOutput.logits.shape == (1, 151936)</code>. Это значит, что когда в Lesson-08 мы подставим реальную модель — весь остальной код останется неизменным.</div></div>


<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-prod' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>⚡</span>Production Demo (Qwen3-8B)</h2>

Теперь заменим MockModel на настоящую **Qwen3-8B** от HuggingFace. Интерфейс остаётся тем же — меняется только источник логитов.


<div style='background:#fff5f5;border-left:4px solid #fc8181;padding:16px 20px;border-radius:0 10px 10px 0;margin:12px 0;color:#2d3748;'><div style='font-weight:700;color:#c53030;margin-bottom:6px;font-size:0.95em;'>⚠️ Внимание</div><div style='line-height:1.7;'>Для этой секции требуется <strong>NVIDIA GPU с VRAM ≥ 16 GB</strong> (Qwen3-8B в BF16 занимает ~16 GB). Если GPU недоступен — ячейки выполнятся в безопасном fallback-режиме и покажут ожидаемую структуру вывода на MockModel.</div></div>


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 load_real_model.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
@lru_cache(maxsize=1)
def _load_qwen3_cached(
    model_id: str,
    dtype_str: str,
) -> Tuple[AutoTokenizer, AutoModelForCausalLM]:
    '''
    Загружает токенизатор и модель Qwen3 из HuggingFace с кэшированием.

    Description:
    ---------------
        При первом вызове загружает модель (~16 GB для 8B BF16).
        При повторных вызовах с теми же аргументами возвращает кэш — мгновенно.
        device_map='auto' автоматически распределяет по доступным GPU.

    Args:
    ---------------
        model_id: Идентификатор модели (например, 'Qwen/Qwen3-8B').
        dtype_str: Тип данных: 'bfloat16' или 'float32'.

    Returns:
    ---------------
        Кортеж (tokenizer, model). model в режиме eval().

    Raises:
    ---------------
        OSError: Если модель не найдена локально и нет интернета.
        RuntimeError: Если недостаточно GPU-памяти.

    Examples:
    ---------------
        >>> tok, mdl = _load_qwen3_cached("Qwen/Qwen3-8B", "bfloat16")
    '''
    # Выбираем torch dtype: bfloat16 экономит ~50% памяти vs float32
    dtype: torch.dtype = (
        torch.bfloat16 if dtype_str == "bfloat16" else torch.float32
    )

    # Загружаем токенизатор — только JSON-конфиг, быстро
    print(f"🔄 Загружаем токенизатор {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, trust_remote_code=True
    )

    # Загружаем веса модели — ~16 GB при первом запуске
    print(f"🔄 Загружаем модель {model_id} ({dtype_str})...")
    print("   Первая загрузка занимает несколько минут...")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto",      # авто-распределение по GPU
        trust_remote_code=True,
    )
    # Переключаем в inference mode: отключает dropout, экономит память
    model.eval()

    print(f"✅ {model_id} загружена")
    return tokenizer, model


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 run_prefill_real.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
def run_prefill_real(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
) -> PrefillOutput:
    '''
    Выполняет prefill с реальной Qwen3-8B.

    Description:
    ---------------
        Токенизирует промпт → forward pass → извлекает логиты последней позиции.
        Логиты возвращаются на CPU в float32 для последующего анализа.
        Интерфейс идентичен run_prefill() — только model другой тип.

    Args:
    ---------------
        model: AutoModelForCausalLM в режиме eval, загруженная на GPU.
        tokenizer: AutoTokenizer для преобразования text → token IDs.
        prompt: Текстовый промпт для обработки.

    Returns:
    ---------------
        PrefillOutput(logits=[1, V], input_len=S).

    Raises:
    ---------------
        ValueError: Если промпт пустой.
        RuntimeError: Если GPU недоступен или недостаточно памяти.

    Examples:
    ---------------
        >>> out = run_prefill_real(model, tokenizer, "Hello")
        >>> assert out.logits.shape[1] == tokenizer.vocab_size
    '''
    # Проверяем непустой промпт
    if not prompt.strip():
        raise ValueError("Промпт не может быть пустым")

    # Токенизируем промпт: text → input_ids [1, S]
    encoding = tokenizer(prompt, return_tensors="pt")
    input_ids: torch.Tensor = encoding["input_ids"]   # [1, S] на CPU

    # Определяем устройство модели для правильного переноса input
    model_device: torch.device = next(model.parameters()).device
    # Переносим input_ids на то же устройство что и модель (GPU)
    input_ids_gpu: torch.Tensor = input_ids.to(model_device)

    print(f"📝 Промпт: '{prompt}'")
    print(f"📊 Токенов: {input_ids.shape[1]}, устройство: {model_device}")
    print("⚙️  Запускаем prefill...")

    # Forward pass без вычисления градиентов — экономим GPU-память
    with torch.no_grad():
        # model() возвращает CausalLMOutputWithPast
        outputs = model(input_ids_gpu)
        # outputs.logits: [B=1, S, V] — логиты для каждой позиции
        logits_all: torch.Tensor = outputs.logits
        # Извлекаем только последнюю позицию: [1, S, V] → [1, V]
        last_logits: torch.Tensor = logits_all[:, -1, :]

    # Возвращаем на CPU в float32 для анализа (BF16 → float32 расширение)
    return PrefillOutput(
        logits=last_logits.cpu().float(),
        input_len=input_ids.shape[1],
    )


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 visualize_top_tokens.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
def visualize_top_tokens(
    prefill_out: PrefillOutput,
    tokenizer: AutoTokenizer,
    top_k: int = TOP_K_DISPLAY,
    title: str = "Top токены после prefill",
) -> None:
    '''
    Визуализирует top-k наиболее вероятных следующих токенов.

    Description:
    ---------------
        Строит горизонтальный bar chart: токен → вероятность (%).
        Цвета: золото (топ-1), фиолет (топ 2-3), серый (остальные).

    Args:
    ---------------
        prefill_out: PrefillOutput с логитами [1, V] или [V].
        tokenizer: AutoTokenizer для декодирования индексов в строки.
        top_k: Количество токенов для отображения.
        title: Заголовок графика.

    Returns:
    ---------------
        None. Отображает matplotlib figure.

    Raises:
    ---------------
        ValueError: Если top_k > vocab_size.

    Examples:
    ---------------
        >>> visualize_top_tokens(prefill_out, tokenizer, top_k=10)
    '''
    # Нормализуем форму логитов: поддерживаем [V] и [1, V]
    logits: torch.Tensor = prefill_out.logits
    if logits.dim() == 2:
        logits = logits[0]   # [1, V] → [V]

    # Вычисляем вероятности: logits → softmax → probabilities
    probs: torch.Tensor = torch.softmax(logits.float(), dim=-1)   # [V]

    # Извлекаем top-k: значения и индексы
    top_probs, top_indices = torch.topk(probs, k=top_k)   # [top_k] каждый

    # Декодируем индексы в читаемые строки токенов
    token_labels: List[str] = [
        repr(tokenizer.decode([int(idx)]))
        for idx in top_indices
    ]

    # Создаём styled figure
    fig, ax = plt.subplots(figsize=(11, 5))
    fig.patch.set_facecolor("#f8f9ff")
    ax.set_facecolor("#f8f9ff")

    # Цветовая схема: топ-1 → primary, топ 2-3 → secondary, остальные → серый
    colors = [
        "#667eea" if i == 0 else "#764ba2" if i < 3 else "#a0aec0"
        for i in range(top_k)
    ]
    bars = ax.barh(
        range(top_k),
        top_probs.numpy() * 100,
        color=colors,
        alpha=0.85,
        height=0.7,
    )

    # Настройка осей
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(token_labels, fontsize=10, fontfamily="monospace")
    ax.invert_yaxis()   # самый вероятный токен сверху
    ax.set_xlabel("Вероятность (%)", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold", color="#2d3748", pad=15)

    # Подписи значений на барах
    for bar, prob_val in zip(bars, top_probs.numpy()):
        ax.text(
            bar.get_width() + 0.002,
            bar.get_y() + bar.get_height() / 2,
            f"{prob_val * 100:.3f}%",
            va="center",
            fontsize=9,
            color="#4a5568",
        )

    # Убираем лишние рамки для чистого вида
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.show()
    print(f"📊 Топ-1: {token_labels[0]} ({top_probs[0].item()*100:.3f}%)")


<div style='background:#1a202c;color:#a0aec0;padding:8px 16px;border-radius:10px 10px 0 0;margin-bottom:-15px;font-family:monospace;font-size:0.8em;display:flex;justify-content:space-between;align-items:center;'><span>🐍 production_demo.py</span><span style='background:#2d3748;padding:2px 10px;border-radius:10px;font-size:0.85em;'>Python</span></div>


In [ ]:
# ── Production Demo ──────────────────────────────────────────────────
DEMO_PROMPT: str = "Что такое KV-кэш в языковых моделях?"

if torch.cuda.is_available():
    # Выводим информацию о GPU
    gpu_name: str = torch.cuda.get_device_name(0)
    gpu_mem_gb: float = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}  ({gpu_mem_gb:.1f} GB VRAM)")

    # Предупреждаем если памяти может не хватить
    if gpu_mem_gb < 14.0:
        warnings.warn(
            f"⚠️  {gpu_mem_gb:.1f} GB может быть недостаточно для Qwen3-8B (~16 GB). "
            "Рассмотрите Qwen/Qwen3-0.6B.",
            RuntimeWarning,
            stacklevel=1,
        )

    # Загружаем модель (первый вызов — несколько минут; повторный — мгновенно)
    tokenizer_real, model_real = _load_qwen3_cached(DEFAULT_MODEL_ID, "bfloat16")

    # Выполняем реальный prefill
    prefill_real: PrefillOutput = run_prefill_real(
        model_real, tokenizer_real, DEMO_PROMPT
    )
    print(f"✅ logits.shape={prefill_real.logits.shape}, "
          f"input_len={prefill_real.input_len}")

    # Визуализируем top-10 токенов
    visualize_top_tokens(
        prefill_real,
        tokenizer_real,
        top_k=TOP_K_DISPLAY,
        title=f"Qwen3-8B Prefill · Top-{TOP_K_DISPLAY} следующих токенов",
    )

else:
    # Fallback: показываем структуру вывода на MockModel
    print("⚠️  GPU не обнаружен — Production Demo в fallback-режиме.")
    print("   На GPU произошло бы следующее:")
    print("   1. Загрузка Qwen3-8B BF16 (~16 GB в VRAM)")
    print("   2. tokenizer(prompt) → input_ids [1, S]")
    print("   3. model(input_ids).logits → [1, S, 151936]")
    print("   4. logits[:, -1, :] → [1, 151936]  (последняя позиция)")
    print("   5. top-10 токенов по вероятности → bar chart")
    print()

    # Демонстрируем структуру с MockModel
    fb_cfg = MockModelConfig(vocab_size=QWEN3_VOCAB_SIZE, hidden_size=64, seed=0)
    fb_model = MockModel(fb_cfg)
    fb_ids = torch.tensor([[151644, 872, 198, 102853, 99355]])
    fb_out = run_prefill(fb_model, fb_ids)
    print(f"📦 PrefillOutput (MockModel fallback):")
    print(f"   logits.shape = {fb_out.logits.shape}  ← такая же форма с Qwen3-8B")
    print(f"   input_len    = {fb_out.input_len}")


<div style='background:#f0fff4;border-left:4px solid #48bb78;padding:16px 20px;border-radius:0 10px 10px 0;margin:12px 0;color:#2d3748;'><div style='font-weight:700;color:#276749;margin-bottom:6px;font-size:0.95em;'>✅ Результат</div><div style='line-height:1.7;'>С реальной Qwen3-8B <code>PrefillOutput.logits.shape == (1, 151936)</code> — точно такая же форма, как у MockModel. График покажет топ-токены с реальными языковыми вероятностями: для вопроса «Что такое KV-кэш?» модель предскажет слова вроде «ключ», «кэш», «память».</div></div>


<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<h2 id='section-nanoinfer' style='color:#2d3748;font-size:1.8em;font-weight:700;border-bottom:3px solid #667eea;padding-bottom:12px;margin-top:40px;margin-bottom:20px;letter-spacing:-0.01em;'><span style='background:linear-gradient(135deg,#667eea,#764ba2);-webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-right:8px;'>🗺️</span>Соответствие nano-infer</h2>

Каждый компонент этого урока имеет прямой аналог в реальном движке `mini-sglang`.

<table style='width:100%;border-collapse:collapse;margin:20px 0;border-radius:12px;overflow:hidden;box-shadow:0 2px 10px rgba(0,0,0,0.06);'>
  <thead><tr style='background:linear-gradient(135deg,#667eea,#764ba2);'>
    <th style='padding:14px 18px;text-align:left;color:white;font-weight:600;'>Урок 03 (Mock)</th>
    <th style='padding:14px 18px;text-align:left;color:white;font-weight:600;'>nano-infer (реальный)</th>
    <th style='padding:14px 18px;text-align:left;color:white;font-weight:600;'>Файл</th>
  </tr></thead>
  <tbody>
    <tr style='background:#f8f9ff;'>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;'><code>MockModel.forward()</code></td>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;'><code>LlamaForCausalLM.forward()</code></td>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;color:#718096;'><code>models/llama.py</code></td>
    </tr>
    <tr style='background:#ffffff;'>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;'><code>PrefillOutput.logits</code></td>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;'>return value of <code>Engine.forward_batch()</code></td>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;color:#718096;'><code>engine/engine.py</code></td>
    </tr>
    <tr style='background:#f8f9ff;'>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;'><code>run_prefill()</code></td>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;'><code>Scheduler._forward()</code> при <code>phase=prefill</code></td>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;color:#718096;'><code>scheduler/scheduler.py</code></td>
    </tr>
    <tr style='background:#ffffff;'>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;'><code>input_ids [B, S]</code></td>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;'><code>batch.input_ids</code> из <code>token_pool[mapping]</code></td>
      <td style='padding:12px 18px;border-bottom:1px solid #e2e8f0;color:#718096;'><code>core.py</code> + <code>scheduler/</code></td>
    </tr>
    <tr style='background:#f8f9ff;'>
      <td style='padding:12px 18px;'><em>(нет KV cache)</em></td>
      <td style='padding:12px 18px;'><code>ctx.kv_cache.store_kv(k, v, out_loc, layer)</code></td>
      <td style='padding:12px 18px;color:#718096;'><code>kvcache/base.py</code> → Lesson-05</td>
    </tr>
  </tbody>
</table>

<div style='background:#f8f9ff;border-left:4px solid #667eea;padding:16px 20px;border-radius:0 10px 10px 0;margin:12px 0;color:#2d3748;'><div style='font-weight:700;color:#667eea;margin-bottom:6px;font-size:0.95em;'>🔗 См. также</div><div style='line-height:1.7;'>Исходный код: <code>nano-infer/python/minisgl/engine/engine.py</code>, <code>nano-infer/python/minisgl/scheduler/scheduler.py</code>, <code>nano-infer/python/minisgl/core.py</code>. В Lesson-04 мы реализуем <strong>decode loop</strong> — авторегрессивную генерацию токенов с использованием этого prefill.</div></div>

<div style='background:#fffff0;border:1px solid #ecc94b40;border-radius:14px;padding:24px 28px;margin:20px 0;'><h4 style='color:#975a16;margin:0 0 14px 0;font-size:1.1em;font-weight:700;'>🤔 Вопросы для самопроверки</h4><ol style='color:#2d3748;line-height:2;margin:0;padding-left:20px;'><li>Почему prefill обрабатывает все токены промпта параллельно, а decode — только один токен за шаг?</li><li>Что случится с формой выходного тензора, если передать в <code>prefill()</code> батч из 3 последовательностей разной длины? Как это решается в реальных движках?</li><li>В чём ключевое отличие <code>MockModel</code> от реального трансформера с точки зрения данных? Почему batch independence гарантирована и там, и там?</li></ol></div>


<div style='text-align:center;margin:30px 0;'><div style='display:inline-block;width:60px;height:3px;background:linear-gradient(to right,transparent,#667eea,transparent);border-radius:2px;'></div></div>
<div style='background:linear-gradient(135deg,rgba(102,126,234,0.08),rgba(118,75,162,0.08));border:1px solid rgba(102,126,234,0.25);border-radius:16px;padding:30px;margin-top:30px;'>
  <h3 style='color:#667eea;margin:0 0 18px 0;font-size:1.3em;font-weight:700;'>🎯 Ключевые выводы</h3>
  <ul style='color:#2d3748;line-height:2;list-style:none;padding:0;margin:0;'>
    <li style='padding:6px 0;border-bottom:1px solid rgba(102,126,234,0.1);'>✅ <strong>Prefill</strong> — первый forward pass по всему промпту; возвращает логиты [B, V] для первого нового токена</li>
    <li style='padding:6px 0;border-bottom:1px solid rgba(102,126,234,0.1);'>✅ <strong>MockModel</strong> воспроизводит <em>форму и интерфейс</em> реальной LM без GPU и 16 GB весов; полезна для тестирования и изучения</li>
    <li style='padding:6px 0;border-bottom:1px solid rgba(102,126,234,0.1);'>✅ <strong>PrefillOutput</strong> содержит логиты + input_len; в nano-infer сюда добавится kv_cache_handle (Lesson-05)</li>
    <li style='padding:6px 0;border-bottom:1px solid rgba(102,126,234,0.1);'>✅ <strong>Batch independence</strong>: результат одной последовательности не зависит от других в батче — верно и для трансформеров</li>
    <li style='padding:6px 0;border-bottom:1px solid rgba(102,126,234,0.1);'>✅ Реальный prefill (Qwen3-8B): <code>model(input_ids).logits[:, -1, :]</code> — тот же интерфейс, другой источник логитов</li>
    <li style='padding:6px 0;'>✅ Следующий шаг: <strong>Lesson-04 — Decode Loop</strong> — авторегрессивная генерация с использованием этого prefill</li>
  </ul>
</div>

<div style='text-align:center;padding:30px 20px;margin-top:40px;border-top:2px solid #e2e8f0;color:#718096;font-size:0.85em;'>
  <p style='margin:0 0 8px 0;'>📘 <strong style='color:#667eea;'>CookBook Series</strong> — inference-from-scratch · Урок 03: Mock-модель и Prefill</p>
  <p style='margin:0;'>Сгенерировано 2026-03-23 · @Verbasik · Следующий урок: <strong>Lesson-04 — Decode Loop</strong></p>
</div>